# Prediction Boundary
### Reproduce the numbers behind a delayed optical-control comparison

**Noether Labs · 18 November 2026**

A telescope's correction reaches its mirror after the atmosphere has moved. Predicting that motion can help. Whether it helps enough depends on the operating condition and the evidence used to compare controllers.

This notebook reconstructs the primary statistics from **600 held-out simulated trials**. It checks all **120 predeclared comparisons**, plots the twelve predictor-versus-integrator conditions, and works through one result: about **49.10 → 42.75 nm** of residual wavefront error in the clean, low-flux, faster-wind condition.

Choose **Runtime → Run all** in Colab. A CPU runtime is sufficient. The only download is a **1.35 MB JSON report** at a fixed Git commit; its SHA-256 fingerprint must match before any calculation. No API keys, GPU, package installation or raw-dataset download is required in a standard Colab Python environment.

This is a reconstruction of statistics from published per-trial metrics. It does **not** rerun the optical simulation or reconstruct those metrics from raw wavefronts. The separate physical audit and complete raw release are linked at the end.

## 1. Obtain the exact report

The revision and fingerprint below identify a specific published result. If either the source changes or a transfer corrupts the bytes, the notebook stops. The filename alone is not the evidence identity.

In [ ]:
import hashlib
import itertools
import json
import math
import statistics
import urllib.request

import matplotlib.pyplot as plt
from scipy.stats import t as student_t

REVISION = "3be53dfc1f1340d0c120c7bd5d600ea6b80f48f0"
REPORT_URL = (
    f"https://raw.githubusercontent.com/noetherlabs/noether-ao/{REVISION}/"
    "results/controller_study_v1/report.json"
)
REPORT_SHA256 = "d0f0ee69bcc19cd5aa0408197343cb2155508237f28692b857a5fa728ddf06be"
MAX_REPORT_BYTES = 2 * 1024**2


def decode_report(payload):
    if not isinstance(payload, bytes) or len(payload) > MAX_REPORT_BYTES:
        raise ValueError("The report is absent or exceeds the 2 MiB download limit.")
    if hashlib.sha256(payload).hexdigest() != REPORT_SHA256:
        raise ValueError("Report fingerprint differs. Stop; do not substitute another revision.")
    return json.loads(payload)


with urllib.request.urlopen(REPORT_URL, timeout=30) as response:
    report_bytes = response.read(MAX_REPORT_BYTES + 1)
report = decode_report(report_bytes)
print(f"Verified {len(report_bytes):,} bytes at commit {REVISION[:12]}.")


## 2. What is being compared?

Five selected controllers were evaluated in twelve conditions: two photon levels × two wind multipliers × three fault settings. Each condition has the same ten paired atmosphere/detector seed identities. This gives 5 × 12 × 10 = 600 trials, but only **ten independent seed units within a condition**. Frames and controller variants are not extra independent replicates.

The episode contains 512 frames. Its primary score uses frames 128–511. Faults begin at frame 256; the primary score therefore includes the earlier part of the episode by design. We do not move that window after looking at the result.

For run $s$, the reported wavefront error is

$$
W_s = \frac{750\,\mathrm{nm}}{2\pi}
\sqrt{\frac{1}{384}\sum_{t=128}^{511}\operatorname{Var}_{\mathrm{pupil}}[\phi_{s,t}]}.
$$

Here $\phi$ is residual phase in radians. This is the root of mean phase variance, rather than the mean of framewise RMS errors. This notebook starts from the retained $W_s$ values.

For a pair of controllers, define $d_s=W_{s,\mathrm{left}}-W_{s,\mathrm{right}}$. The fixed comparison reports

$$
\bar d \;\pm\; t_{9,\,1-0.05/(2\times120)}\;\frac{s_d}{\sqrt{10}}.
$$

There are $\binom{5}{2}\times12=120$ comparisons. Keeping that family size matters even when we display only twelve. Bonferroni does not require independence between comparisons. The Student-t interval is exact for independent normal seed differences; with ten seeds and a nonnormal distribution, its coverage is approximate.

An interval excluding zero is only part of the decision. Every trial of the proposed winner must also have finite outputs, improve on its zero-mirror reference, and stay within the 1% saturation bound. We reconstruct those checks rather than treating a positive mean as sufficient evidence.

The code below validates complete trial and comparison identities. It then recomputes each mean, interval and winner from per-trial metrics, checking the stored paired differences as well. A small floating-point tolerance applies to arithmetic; the report itself still requires an exact byte fingerprint.

In [ ]:
CONTROLLERS = ("integrator", "predictive", "forward", "fixed", "learned")
PHOTONS = (1000.0, 1000000.0)
WINDS = (0.5, 2.0)
FAULTS = ("none", "optical_gain", "vibration")
SEEDS = tuple(range(23001, 23011))
FAMILY_SIZE = 120
FAMILY_ALPHA = 0.05
SATURATION_LIMIT = 0.01
CONDITIONS = tuple(itertools.product(PHOTONS, WINDS, FAULTS))
PAIRS = tuple(itertools.combinations(CONTROLLERS, 2))
EXPECTED_TRIALS = set(itertools.product(CONTROLLERS, PHOTONS, WINDS, FAULTS, SEEDS))
EXPECTED_COMPARISONS = {
    (left, right, photons, wind, fault)
    for left, right in PAIRS
    for photons, wind, fault in CONDITIONS
}

# These tolerances apply only to the independently recomputed floating-point
# statistics. Authentication above always requires the exact original bytes.
STAT_ATOL_NM = 1e-9
STAT_RTOL = 1e-11


def finite_number(value):
    return type(value) in (int, float) and math.isfinite(value)


def index_trials(document):
    if (document.get("status") != "complete_descriptive_report"
            or document.get("study_id") != "controller_study_v1"
            or document.get("post_burn_frames") != [128, 512]
            or document.get("primary_pairwise_family") != FAMILY_SIZE
            or document.get("familywise_alpha") != FAMILY_ALPHA
            or document.get("evaluation_records") != len(EXPECTED_TRIALS)):
        raise ValueError("The report does not describe the frozen primary study.")
    trials = document.get("evaluation_trials")
    if type(trials) is not list or len(trials) != len(EXPECTED_TRIALS):
        raise ValueError("Exactly 600 held-out trial records are required.")
    indexed = {}
    for trial in trials:
        case = trial["case"]
        controller = case["candidate"]["controller"]
        photons, wind, fault, seed = (case[k] for k in ("photons", "wind_scale", "fault", "seed"))
        if (type(controller) is not str or type(fault) is not str
                or type(seed) is not int or not finite_number(photons)
                or not finite_number(wind) or case.get("stage") != "evaluation"
                or type(case.get("detector_seed")) is not int
                or case["detector_seed"] != seed + 100000):
            raise ValueError("Malformed evaluation identity or detector pairing.")
        key = controller, photons, wind, fault, seed
        if key not in EXPECTED_TRIALS or key in indexed:
            raise ValueError("Unexpected or duplicate held-out trial identity.")
        expected_id = "predictive-01" if controller == "predictive" else controller + "-04"
        if case["candidate"]["id"] != expected_id or trial.get("status") != "completed":
            raise ValueError("Wrong selected candidate or incomplete trial.")
        metric = trial["post_burn"]
        if (type(metric.get("all_outputs_finite")) is not bool
                or not all(finite_number(metric.get(k))
                           for k in ("wfe_nm", "open_wfe_nm", "saturation_fraction"))
                or metric["wfe_nm"] < 0 or metric["open_wfe_nm"] <= 0
                or not 0 <= metric["saturation_fraction"] <= 1):
            raise ValueError("Missing or invalid primary trial metrics.")
        indexed[key] = metric
    if set(indexed) != EXPECTED_TRIALS:
        raise ValueError("The declared held-out cohort is incomplete.")
    return indexed


def candidate_eligible(indexed, controller, condition):
    rows = [indexed[(controller, *condition, seed)] for seed in SEEDS]
    return all(row["all_outputs_finite"] is True
               and row["wfe_nm"] < row["open_wfe_nm"]
               and row["saturation_fraction"] <= SATURATION_LIMIT for row in rows)


def paired_summary(indexed, left, right, condition):
    differences = [indexed[(left, *condition, seed)]["wfe_nm"]
                   - indexed[(right, *condition, seed)]["wfe_nm"] for seed in SEEDS]
    mean = statistics.mean(differences)
    sd = statistics.stdev(differences)
    critical = float(student_t.ppf(1 - FAMILY_ALPHA / (2 * FAMILY_SIZE), len(SEEDS) - 1))
    half_width = critical * sd / math.sqrt(len(SEEDS))
    low, high = mean - half_width, mean + half_width
    winner = None
    if low > 0 and candidate_eligible(indexed, right, condition):
        winner = right
    elif high < 0 and candidate_eligible(indexed, left, condition):
        winner = left
    return dict(mean_nm=mean, low_nm=low, high_nm=high, n=len(SEEDS),
                paired_differences_nm=differences, winner=winner)


def close(actual, expected, label):
    if not (finite_number(actual) and finite_number(expected)
            and math.isclose(actual, expected, abs_tol=STAT_ATOL_NM, rel_tol=STAT_RTOL)):
        raise ValueError(f"Reconstruction differs: {label}.")


def reproduce(document):
    indexed = index_trials(document)
    rows = document.get("comparisons")
    if type(rows) is not list or len(rows) != FAMILY_SIZE:
        raise ValueError("All 120 predeclared comparisons are required.")
    reconstructed = {}
    for saved in rows:
        key = saved["left"], saved["right"], saved["photons"], saved["wind_scale"], saved["fault"]
        if key not in EXPECTED_COMPARISONS or key in reconstructed:
            raise ValueError("Unexpected or duplicate comparison identity.")
        if saved["difference"] != "left_minus_right":
            raise ValueError("Comparison sign convention changed.")
        primary = saved["post_burn"]
        if (primary["status"] != "complete" or primary["family_size"] != FAMILY_SIZE
                or primary["inference"] != "bonferroni_simultaneous"
                or primary["interval"]["n"] != len(SEEDS)):
            raise ValueError("Primary comparison rule changed.")
        computed = paired_summary(indexed, key[0], key[1], key[2:])
        paired = primary["paired_differences_nm"]
        if type(paired) is not list or len(paired) != len(SEEDS):
            raise ValueError("All ten paired differences are required.")
        for actual, expected in zip(computed["paired_differences_nm"], paired):
            close(actual, expected, "paired WFE difference")
        for field in ("mean_nm", "low_nm", "high_nm"):
            close(computed[field], primary["interval"][field], field)
        close(primary["interval"]["confidence"], 1 - FAMILY_ALPHA / FAMILY_SIZE,
              "marginal confidence")
        if computed["winner"] != primary["winner"]:
            raise ValueError("Interval and eligibility gates do not reproduce the retained winner.")
        reconstructed[key] = computed
    if set(reconstructed) != EXPECTED_COMPARISONS:
        raise ValueError("Comparison family is incomplete.")
    return indexed, reconstructed


trials, comparisons = reproduce(report)
print(f"Reconstructed {len(trials)} held-out trials and all {len(comparisons)} primary contrasts.")
print(f"Statistics matched using max({STAT_ATOL_NM:g} nm, {STAT_RTOL:g} × magnitude) tolerance.")


## 3. Keep the whole operating envelope visible

Positive differences below favor the predictor. “Uncertain” means the frozen procedure did not establish a winner; it does not mean the controllers are equivalent. The same shared seed set is reused across conditions.

In [ ]:
FAULT_LABELS = {"none": "Clean", "optical_gain": "Optical gain", "vibration": "Vibration"}
prediction = [comparisons[("integrator", "predictive", *condition)] for condition in CONDITIONS]
print("Photons    Wind  Condition       Integrator - predictor [adjusted interval], nm")
for condition, row in zip(CONDITIONS, prediction):
    photons, wind, fault = condition
    verdict = "predictive advantage" if row["winner"] == "predictive" else "uncertain"
    print(f"{photons:9,.0f}  {wind:3.1f}  {FAULT_LABELS[fault]:14s}"
          f" {row['mean_nm']:7.2f} [{row['low_nm']:7.2f}, {row['high_nm']:7.2f}]  {verdict}")
supported_prediction = sum(row["winner"] == "predictive" for row in prediction)
learned_beats_both = sum(
    comparisons[("integrator", "learned", *condition)]["winner"] == "learned"
    and comparisons[("forward", "learned", *condition)]["winner"] == "learned"
    for condition in CONDITIONS
)
print(f"\nPredictor supported against integrator: {supported_prediction}/12 conditions.")
print(f"Learned mixture supported against both integrator and forward: {learned_beats_both}/12.")


In [ ]:
plt.rcParams.update({"font.family": "STIXGeneral", "mathtext.fontset": "stix",
                     "font.size": 12, "axes.spines.top": False,
                     "axes.spines.right": False, "svg.fonttype": "path"})
figure, axes = plt.subplots(1, 2, figsize=(12, 22 / 3), sharex=True, sharey=True)
condition_labels = [f"Wind {wind:g} / {FAULT_LABELS[fault]}" for wind in WINDS for fault in FAULTS]
for ax, photons in zip(axes, PHOTONS):
    ax.axvline(0, color="0.55", linestyle="--", linewidth=0.8)
    for y, (wind, fault) in enumerate(itertools.product(WINDS, FAULTS)):
        row = comparisons[("integrator", "predictive", photons, wind, fault)]
        color = "#266a83" if row["winner"] == "predictive" else "#636363"
        ax.plot([row["low_nm"], row["high_nm"]], [y, y], color=color, linewidth=1.5)
        ax.plot(row["mean_nm"], y, "o", color=color, markersize=4)
    ax.set_title(f"{photons:,.0f} photons / frame", loc="left")
    ax.set_xlabel("Integrator − predictor WFE (nm)")
    ax.set_yticks(range(6), condition_labels)
    ax.tick_params(axis="y", length=0)
axes[0].invert_yaxis()
figure.suptitle("Where prediction helps", fontsize=17, x=0.02, ha="left", y=0.98)
figure.text(0.02, 0.92, "All twelve conditions from the completed simulated comparison", fontsize=12)
figure.text(0.02, 0.04,
            "Positive values favor prediction. Ten paired seeds per condition; intervals retain the full 120-comparison correction.\n"
            "Blue: interval and trial eligibility support prediction. Gray: uncertainty remains; this does not establish equivalence.",
            fontsize=10, color="0.3")
figure.subplots_adjust(left=0.19, right=0.98, bottom=0.20, top=0.85, wspace=0.08)
plt.show()


### Nearly the same mean is not the same evidence

In clean, faster-wind operation, the mean integrator-minus-predictor difference is about 6.35 nm at low flux and 6.58 nm at high flux. The conclusions differ because the ten seed differences have very different spreads.

At high flux, seed 23005 contributes a 46.40 nm difference; the other nine lie between about 1.98 and 2.48 nm. The predeclared Student-t interval reflects that spread. We retain all ten seeds and the original interval. This descriptive inspection neither diagnoses the cause of that run nor justifies removing it. With only two tested photon levels, the study also cannot locate a flux threshold at which prediction starts or stops helping.

In [ ]:
seed_figure, seed_axes = plt.subplots(1, 2, figsize=(12, 22 / 3), sharex=True, sharey=True)
seed_examples = []
for ax, photons in zip(seed_axes, PHOTONS):
    row = comparisons[("integrator", "predictive", photons, 2.0, "none")]
    differences = row["paired_differences_nm"]
    seed_examples.append(row)
    ax.axvline(0, color="0.55", linestyle="--", linewidth=0.8)
    ax.plot(differences, range(10), "o", color="#266a83", markersize=4)
    ax.plot([row["low_nm"], row["high_nm"]], [11, 11], color="#222222", linewidth=1.5)
    ax.plot(row["mean_nm"], 11, "D", color="#222222", markersize=5)
    ax.axhline(10, color="0.85", linewidth=0.7)
    ax.set_title(f"{photons:,.0f} photons / frame", loc="left", pad=12)
    ax.set_xlabel("Integrator − predictor WFE (nm)")
    ax.set_yticks([*range(10), 11], [*map(str, SEEDS), "Mean + interval"])
    ax.tick_params(axis="y", length=0)
seed_axes[0].invert_yaxis()
seed_axes[0].set_ylabel("Paired seed", labelpad=10)
seed_figure.suptitle("Similar means, different uncertainty", fontsize=17,
                    x=0.02, ha="left", y=0.98)
seed_figure.text(0.02, 0.92, "Clean operation, wind multiplier 2; every evaluation seed retained", fontsize=12)
seed_figure.text(0.02, 0.04,
                 "Circles: individual paired differences. Diamonds: means, with the original 120-comparison-adjusted intervals.\n"
                 "Both panels use the same horizontal scale. No seed is removed or downweighted.",
                 fontsize=10, color="0.3")
seed_figure.subplots_adjust(left=0.16, right=0.98, bottom=0.20, top=0.85, wspace=0.08)
plt.show()
for photons, row in zip(PHOTONS, seed_examples):
    print(f"{photons:,.0f} photons: mean {row['mean_nm']:.6f} nm; "
          f"interval [{row['low_nm']:.6f}, {row['high_nm']:.6f}] nm")


## 4. Work through one condition

Take 1,000 photons per frame, wind multiplier 2, and no injected fault. We calculate the two means and their paired difference directly. The percentage below divides the **difference of controller means** by the integrator mean. It is neither the mean of ten percentage improvements nor a pooled effect across the instrument's operating envelope.

In [ ]:
condition = (1000.0, 2.0, "none")
integrator_mean = statistics.mean(trials[("integrator", *condition, seed)]["wfe_nm"] for seed in SEEDS)
predictive_mean = statistics.mean(trials[("predictive", *condition, seed)]["wfe_nm"] for seed in SEEDS)
example = comparisons[("integrator", "predictive", *condition)]
percent_of_integrator_mean = 100 * (integrator_mean - predictive_mean) / integrator_mean
print(f"Integrator mean: {integrator_mean:.6f} nm")
print(f"Predictor mean:  {predictive_mean:.6f} nm")
print(f"Paired difference: {example['mean_nm']:.6f} nm")
print(f"Adjusted interval: [{example['low_nm']:.6f}, {example['high_nm']:.6f}] nm")
print(f"Difference / integrator mean: {percent_of_integrator_mean:.3f}%")


The complete comparison supports prediction over the integrator in five of twelve conditions, all at the lower photon level. The remaining seven comparisons are uncertain. The learned mixture does not beat both the integrator and the nonlinear forward controller in any condition under the frozen rule.

That pattern is a reason to investigate a particular operating region. It does not isolate why the region exists. Temporal prediction, measurement noise, controller tuning and finite-sample uncertainty all enter this experiment. A causal explanation would require a separate ablation.

The learned result is also specific to the implementation and training budget: one initialization and a finite optimization run. It does not rule out a better-trained mixture or a different learned controller. Some statistically separated controller differences in the full family are only hundredths of a nanometre; the protocol did not declare a minimum improvement worth adopting on an instrument.

## 5. Try to break the reconstruction

These small checks make the notebook refuse missing or duplicate trials, a new seed, altered metrics, duplicate comparisons, a reduced comparison family, or a changed report. The last check leaves a favorable numerical interval intact but makes one proposed-winner trial exceed the saturation bound. It must withhold that winner.

The copies exist only in memory. Nothing rewrites the downloaded evidence.

In [ ]:
from copy import deepcopy


def must_refuse(label, operation):
    try:
        operation()
    except ValueError:
        print(f"Refused: {label}")
    else:
        raise AssertionError(f"Accepted corrupted evidence: {label}")


must_refuse("altered report bytes", lambda: decode_report(report_bytes + b" "))
changed = deepcopy(report)
changed["evaluation_trials"].pop()
must_refuse("missing trial", lambda: reproduce(changed))
changed = deepcopy(report)
changed["evaluation_trials"][0] = deepcopy(changed["evaluation_trials"][1])
must_refuse("duplicate trial", lambda: reproduce(changed))
changed = deepcopy(report)
changed["evaluation_trials"][0]["case"]["seed"] = 999999
changed["evaluation_trials"][0]["case"]["detector_seed"] = 1099999
must_refuse("undeclared seed", lambda: reproduce(changed))
changed = deepcopy(report)
changed["evaluation_trials"][0]["post_burn"]["wfe_nm"] += 0.1
must_refuse("changed per-trial metric", lambda: reproduce(changed))
changed = deepcopy(report)
changed["comparisons"][0] = deepcopy(changed["comparisons"][1])
must_refuse("duplicate comparison", lambda: reproduce(changed))
changed = deepcopy(report)
changed["primary_pairwise_family"] = 12
must_refuse("smaller correction selected after seeing results", lambda: reproduce(changed))

# A favorable interval alone is insufficient. The intervention must also pass
# the frozen per-trial usefulness and actuator-saturation checks.
changed = deepcopy(report)
for trial in changed["evaluation_trials"]:
    case = trial["case"]
    if (case["candidate"]["controller"] == "predictive"
            and (case["photons"], case["wind_scale"], case["fault"]) == condition
            and case["seed"] == SEEDS[0]):
        trial["post_burn"]["saturation_fraction"] = 0.02
        break
ineligible_trials = index_trials(changed)
ineligible = paired_summary(ineligible_trials, "integrator", "predictive", condition)
if ineligible["low_nm"] <= 0 or ineligible["winner"] is not None:
    raise AssertionError("The favorable interval must not override failed eligibility.")
must_refuse("retained winner with newly failed saturation eligibility", lambda: reproduce(changed))
print("All eight evidence-corruption checks passed.")


## What this establishes

The notebook independently reduces the **published per-trial WFE metrics** and checks their statistical consistency with the published comparisons. It does not establish physical fidelity of the simulated wavefront sensor, independently generate the atmosphere, or measure hardware performance. A simulated 1 ms time step is not a measured 1 ms control deadline.

The full raw release contains sensor, phase, command and training arrays. Reconstructing WFE from those arrays is a different and larger exercise. The retained physical audit checks phase variance, coherent pupil Strehl, command delay and modal bounds using a separate NumPy reduction, while sharing the saved basis and atmospheric streams. These checks should not be described as on-sky validation.

- [Article: Prediction Boundary](https://www.noether.one/research/prediction-boundary)
- [Published report and all comparisons, fixed revision](https://github.com/noetherlabs/noether-ao/tree/3be53dfc1f1340d0c120c7bd5d600ea6b80f48f0/results/controller_study_v1)
- [Frozen study protocol](https://github.com/noetherlabs/noether-ao/blob/3be53dfc1f1340d0c120c7bd5d600ea6b80f48f0/docs/controller_study_v1.md)
- [Physical reconstruction audit](https://github.com/noetherlabs/noether-ao/blob/3be53dfc1f1340d0c120c7bd5d600ea6b80f48f0/results/controller_study_v1/physical-audit.json)
- [Raw dataset, pinned revision](https://huggingface.co/datasets/noetherlabs/prediction-boundary-v1/tree/73234e5a6a1374568179562aab76f9f6caa9354e)

**Citation:** Noether Labs. *Prediction Boundary: Where prediction helps in a delayed optical loop*. 2026. https://www.noether.one/research/prediction-boundary

The two figures are available as notebook variables, <code>figure</code> and <code>seed_figure</code>. To save either in a chosen directory, use Matplotlib's <code>savefig</code> with .svg, .png or .pdf; <code>dpi=150</code> preserves the intended 1800 × 1100 PNG dimensions. Exporting is optional and does not change the calculation.

The raw dataset is distributed under CC-BY-4.0. Project-owned Noether AO code is Apache-2.0. The completed evaluation cohort is public and exposed; it should not be reused as an untouched test set for a subsequently tuned method.